# NB1 · The Before and the After

**Agentic Systems Engineering · Lecture 1 · Vizuara · Dr. Rajat Dandekar**

You are the engineer **Northbridge Cycles** has just hired. Northbridge builds electric bikes in Manchester, about
18,000 a month, and has run its business on **Ledgerline ERP 7.2** since 1998. On Monday 26 October 2026 Emma Clarke in
purchasing needs **2,000 battery packs** (`BAT-500`) from Voltway Cells, Eindhoven, for the spring launch run.
Everything here is fictional: the ERP, the company, the people and the data.

In this notebook you will:
1. see why agents matter in a company like this;
2. meet the ERP: what SAP, Oracle, Dynamics and Odoo are, then Ledgerline from Python, as a **black box**;
3. do the battery pack order **the legacy way**, screen by screen, in code, and measure it;
4. let **Ask Northbridge** do the same missions, one sentence each;
5. look at **what is behind it**: the tool schemas, one tool call by hand, and MCP, the plug;
6. write down a Before from your own work, which is the start of assignment A1.

The model calls run in **replay mode**: they come from our recorded run (cassette `L1v4_after`), so the notebook
runs with no API key and shows exactly what happened. Six exercises are marked `TODO (exercise N)`. Each one has its
solution in place behind `SOLUTION = True`, so the notebook runs top to bottom. Set it to `False` and try it yourself.

In [1]:
# Setup: install the Ledgerline sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Ledgerline sandbox + reference agent)
try:
    import ledgerline  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ledgerline-erp[all] @ git+{COURSE_REPO}.git#subdirectory=ledgerline"], check=True)
    import ledgerline
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("LEDGER_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Ledgerline", ledgerline.__version__, "· model mode:", os.environ["LEDGER_LLM_MODE"])

Ledgerline 0.1.0 · model mode: replay


In [2]:
import json

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)


def gbp(x):
    """285600 -> £285,600"""
    return f"£{x:,.0f}" if float(x) == int(float(x)) else f"£{x:,.2f}"


def show_json(d, n=1200):
    print(json.dumps(d, indent=1, default=str, ensure_ascii=False)[:n])


print(gbp(285600), gbp(250000), gbp(119))

£285,600 £250,000 £119


## 1. Why agents matter

A company like Northbridge runs on a few systems of record. Every order, receipt, invoice, payslip and leave day is
a row in one of them. Those systems are correct, auditable and old. Their way in is a **screen** that expects codes:
a supplier code, a tax code, a cost centre, a document number. A person in purchasing spends much of the day turning
what they want ("2,000 battery packs from Voltway by 9 November") into those codes, in the right order of screens,
and finding out later what went wrong.

An **agent** sits between the person and the system. It takes the sentence, looks the codes up, shows a preview,
waits for a yes, does the work through the system's own rules, and comes back with a receipt. It does not replace
the ERP: it is a new front door to it. This course builds that front door properly: it is safe, measured, and fit for
production.

To know whether it helps, we first **measure the Before**.

## 2. Meet the ERP

**What an ERP is.** An ERP (enterprise resource planning) system is one shared database plus the business rules
for purchasing, stores, finance, sales, production and HR, so that every department works on the same record.
The big four you will meet at work are **SAP** (S/4HANA, the large-company default; transactions such as `ME21N`
to create a purchase order), **Oracle** (E-Business Suite and Oracle Fusion Cloud ERP), **Microsoft Dynamics 365**
(Finance and Supply Chain, and Business Central for smaller firms) and **Odoo** (open source, modular, popular with
mid-sized companies). They differ in scale and price, but they share the same shape: master data (items,
suppliers, people), documents with a status (request → order → receipt → invoice), approval limits, and screens
or APIs that check codes. **Ledgerline ERP 7.2** is our stand-in. It is small enough to run in a notebook and old
enough to have every classic trap.

`ERP()` is one whole Northbridge in memory: items, suppliers, people, cost centres and past orders. Its clock is
set to Monday 26 October 2026. `serve()` puts it behind HTTP, the way Ledgerline's own legacy client reaches it.

A **service** is one operation of the ERP. A flat JSON payload goes in. The legacy **envelope** comes out: `rc`
is S or E, plus `err_cd`, `msg` and `data`. We never look inside a service.

In [3]:
import httpx

from ledgerline import ERP
from ledgerline.erp.receipts import install_receipts
from ledgerline.server import serve

erp = ERP()
install_receipts(erp)          # goods receipt (GRN), the last step of procure-to-pay
url = serve(port=8077, erp=erp)

login = httpx.post(url + "/login", json={"user_id": "ECLARKE", "password": "emma@123", "ou_cd": "NBC-MAN-01"}).json()
H = {"Authorization": "Bearer " + login["token"]}
print(login["msg"])


def svc(name, **payload):
    """Call one Ledgerline service over HTTP, as Emma. Returns the envelope."""
    return httpx.post(f"{url}/svc/{name}", json=payload, headers=H).json()


show_json(svc("stk_query", itm_cd="BAT-500", whs_cd="WH-MAN-STR"))

Ledgerline is running at http://127.0.0.1:8077 · legacy client: http://127.0.0.1:8077/legacy
WELCOME, ECLARKE. LAST SIGN-IN 24-10-2026 18:02
{
 "rc": "S",
 "err_cd": null,
 "msg": "",
 "data": {
  "itm_cd": "BAT-500",
  "whs_cd": "WH-MAN-STR",
  "on_hand": 5400,
  "allocated": 0,
  "on_order": 0,
  "avg_daily_issue": 600,
  "reorder_lvl": 3000,
  "cover_days": 9
 }
}


In [4]:
env = svc("itm_search", q="battery pack 5")
print(env["msg"])
pd.DataFrame(env["data"]["rows"])

4 ROWS FOUND


,itm_cd,descr,uom_cd,active
0,BAT-500,"Battery pack, 36 V 500 Wh",NOS,1
1,BAT-500-R,"Battery pack, 36 V 500 Wh (refurbished)",NOS,1
2,BAT-375,"Battery pack, 36 V 375 Wh",NOS,1
3,BAT-750,"Battery pack, 48 V 750 Wh",NOS,1


Several battery packs look alike. `-R` is refurbished stock and is never ordered for production. A person has to
know that. Here is the whole catalogue of services:

In [5]:
cat = httpx.get(url + "/services").json()
print(len(cat), "services")
pd.DataFrame([{k: s[k] for k in ("name", "kind", "doc")} for s in cat]).head(12)

41 services


,name,kind,doc
0,itm_search,read,ITEM ENQUIRY
1,itm_get,read,ITEM MASTER GET
2,sup_search,read,SUPPLIER ENQUIRY
3,sup_get,read,SUPPLIER MASTER GET
4,sup_create,write,SUPPLIER CREATE
5,sup_cert_get,read,SUPPLIER CERT LIST
6,sup_cert_upload,write,SUPPLIER CERT UPLOAD
7,sup_rate_get,read,RATE CONTRACT GET
8,cc_list,read,COST CENTRE LIST
9,cc_get,read,COST CENTRE GET


### TODO (exercise 1): which Voltway?
Call the supplier search service `sup_search` with `q="Voltway"`. How many rows come back, and which one must
never be used? Put its code in `blocked`. (Look at the other row too: is it the same company?)

In [6]:
SOLUTION = True
blocked = None
if SOLUTION:
    rows = svc("sup_search", q="Voltway")["data"]["rows"]
    display(pd.DataFrame(rows))
    blocked = next(r["sup_cd"] for r in rows if r["status"] == "BLOCKED")
print("blocked duplicate:", blocked)
assert blocked == "VC-EIN-0009"

,sup_cd,name,city,state,status
0,VC-EIN-0042,Voltway Cells,Eindhoven,NL,ACTIVE
1,VC-EIN-0009,"Voltway Cells (old code, do not use)",Eindhoven,NL,BLOCKED
2,VPC-ROT-0107,Voltway Paints & Coatings,Rotterdam,NL,ACTIVE


blocked duplicate: VC-EIN-0009


## 3. The Before, measured

`Desk` is one person at one PC with the legacy client open. It reads the same screen files the client reads
(`erp/screens/*.yaml`), so pressing a button here makes exactly the service call the client would make. `Tally`
counts what the person did: screens opened, fields typed, codes typed, buttons, sign-ins and error codes.

Here is the order-entry screen as a file: its fields, and which service each button calls.

In [7]:
from ledgerline.legacy_ui.driver import Desk, Tally, screen_yaml

print(screen_yaml("PUR-PO-01")[:1500])

code: PUR-PO-01
title: PURCHASE ORDER ENTRY
menu: Purchase > Order entry
roles:
- PUR-EXEC
- PUR-SR
- SYS-ADMIN
help: 'Enter the source request, supplier and item, then press Fetch: it brings in
  the contract rate, tax code, payment terms and the request''s cost centre, date
  and quantity. Save gives the order number. Orders without a request (type DIR) are
  for £50,000 or less.'
fields:
- name: po_no
  label: PO NO
  type: str
  readonly: true
- name: po_type
  label: PO TYPE
  type: str
  req: true
  default: STD
  help: STD/DIR
- name: pr_no
  label: PR NO
  type: str
  req: false
  lookup: pr_search
- name: sup_cd
  label: SUP CD
  type: str
  req: true
  lookup: sup_search
- name: itm_cd
  label: ITEM CD
  type: str
  req: true
  lookup: itm_search
- name: qty
  label: QTY
  type: int
  req: true
- name: rate
  label: RATE
  type: float
  req: true
- name: tax_cd
  label: TAX CD
  type: str
  req: true
  lookup: tax_list
- name: pt_cd
  label: PAY TERM
  type: str
  readonly: t

Now the five missions, the way someone does them the first time: keeping what **Fetch** fills in.

In [8]:
tally = Tally()
http = httpx.Client(base_url=url)


def desk(who, uid, pw):
    return Desk(http, tally, who).sign_in(uid, pw)


marks = {}
def mission(name):
    marks[name] = (dict(tally.c), len(tally.events))


mission("1 stock")
emma = desk("Emma", "ECLARKE", "emma@123")
s = emma.open("INV-ITM-03").type(q="battery pack 5").press("Search").pick(0)
item = s.form["itm_cd"]
s = emma.open("INV-STK-02").type(itm_cd=item).press("Query")
print("stock:", s.form["on_hand"], "on hand,", s.form["avg_daily_issue"], "a day,", s.form["cover_days"], "days of cover (Manchester)")

mission("2 request")
s = emma.open("PUR-PR-01").type(itm_cd=item).press("Fetch")
print("Fetch filled cost centre:", s.form["cc_cd"])
s.type(qty="2000", need_dt="09-11-2026", remarks="spring-launch run").press("Save").press("Submit")
pr_no = s.form["pr_no"]
print(s.status, "| estimate", gbp(float(s.form["est_value"])))

mission("3 approve")
daniel = desk("Daniel", "DOKAFOR", "daniel@123")
s = daniel.open("PUR-PR-04").press("Inbox")
s.pick(next(i for i, r in enumerate(s.rows) if r["doc_no"] == pr_no)).press("Fetch").type(decision="A", comment="ok").press("Confirm")
print(s.status)

mission("4 order")
s = emma.open("PUR-POH-02").type(itm_cd=item).press("Query").pick(0)
sup = s.form["sup_cd"]
s = emma.open("PUR-PO-01").type(pr_no=pr_no, sup_cd=sup, itm_cd=item).press("Fetch")
print("Fetch filled tax code", s.form["tax_cd"], "and cost centre", s.form["cc_cd"])
s.press("Save")
print("Save:", s.status)
s.type(tax_cd="VAT20-RC").press("Save")
po_no = s.form["po_no"]
print("Save again:", s.status, "| status field:", s.form["status"])
s = emma.open("PUR-POR-05").type(po_no=po_no).press("Fetch").press("Release")
print("Release:", s.status)

stock: 5400 on hand, 600 a day, 9 days of cover (Manchester)
Fetch filled cost centre: PRD-05
PR/2026/04417 SUBMITTED FOR APPROVAL | estimate £273,600


PR/2026/04417 APPROVED BY DOKAFOR
Fetch filled tax code VAT20-UK and cost centre PRD-05
Save: ERR-PO-0412: TAX CODE DOES NOT MATCH SUPPLIER COUNTRY (NL)
Save again: PO/2026/01182 SAVED | status field: DR
Release: ERR-PO-0715: DOCUMENT NOT IN RELEASABLE STATUS (DR)


**Saved is not done.** The status bar said SAVED, but the status field says `DR`, a Draft. The order took cost
centre PRD-05 from the request, and PRD-05 (the city e-bike line) closed on 15 October. Release refuses the order. The
fix is order amendment: change the cost centre to PRD-07, then release again. Then Aiko receives the goods in two
deliveries.

In [9]:
s = emma.open("PUR-POM-06").type(po_no=po_no).press("Fetch").type(cc_cd="PRD-07").press("Amend")
print("Amend:", s.status)
s = emma.open("PUR-POR-05").type(po_no=po_no).press("Fetch").press("Release")
print("Release:", s.status)

mission("5 receive")
aiko = desk("Aiko", "ATANAKA", "aiko@123")
for q, note in ((1200, "DN-4471"), (800, "DN-4486")):
    s = aiko.open("INV-GRN-03").type(po_no=po_no).press("Fetch").type(qty=str(q), remarks=note).press("Save")
    print(s.status, "| balance", s.form.get("balance"))
mission("end")

Amend: PO/2026/01182 AMENDED
Release: PO/2026/01182 RELEASED


GRN/2026/30001 SAVED | balance 800
GRN/2026/30002 SAVED | balance 0


The tally, mission by mission. Minutes cannot be measured without a person, so the last column is an
**estimate** from the keystroke-level model: 0.28 s a key, 1.1 s a point-and-click, 1.35 s to think before each
field or button, and 2 s for each screen or service reply. Waiting for Daniel and the phone call to IT are not
included.

In [10]:
def klm_seconds(events):
    t = 0.0
    for e in events:
        if e["action"] == "open":
            t += 1.35 + 2 * 1.1 + 2.0
        elif e["action"] == "type":
            t += 1.35 + 1.1 + 0.28 * len(e["detail"].split(" = ", 1)[-1])
        elif e["action"] in ("press", "select"):
            t += 1.35 + 1.1 + (2.0 if e.get("service") else 0)
    return t


names = list(marks)
cols = ["screens opened", "fields typed", "codes typed", "characters typed", "buttons pressed", "sign-ins", "error codes"]
rows = []
for a, b in zip(names, names[1:]):
    (c0, e0), (c1, e1) = marks[a], marks[b]
    rows.append({"mission": a, **{c: c1.get(c, 0) - c0.get(c, 0) for c in cols},
                 "minutes (estimate)": round(klm_seconds(tally.events[e0:e1]) / 60, 1)})
before = pd.DataFrame(rows).set_index("mission")
before.loc["total"] = before.sum()
before.loc["total", "minutes (estimate)"] = round(klm_seconds(tally.events) / 60, 1)
before

,screens opened,fields typed,codes typed,characters typed,buttons pressed,sign-ins,error codes,minutes (estimate)
mission,,,,,,,,
1 stock,3,4,1,36,4,1,0,0.9
2 request,1,4,1,38,3,0,0,0.7
3 approve,2,4,1,20,5,1,0,0.8
4 order,5,9,9,91,11,0,2,2.0
5 receive,3,8,2,62,5,1,0,1.3
total,14,29,14,247,28,3,2,5.6


### TODO (exercise 2): the careful path
Luis, the senior buyer, knows every trap. He types `PRD-07` on the request and `VAT20-RC` on the order before the
first Save. Re-run missions 2 to 4 his way on a fresh Northbridge. Put the number of codes typed in mission 4 in
`careful_codes`. Knowing the traps removes the errors. Does it remove the steps?

In [11]:
SOLUTION = True
careful_codes = None
if SOLUTION:
    from fastapi.testclient import TestClient
    from ledgerline.server import create_app

    erp2 = ERP(); install_receipts(erp2)
    app2 = create_app(); app2.state.tenants["default"] = erp2
    t2 = Tally(); http2 = TestClient(app2)
    k2 = Desk(http2, t2, "Emma").sign_in("ECLARKE", "emma@123")
    s = k2.open("PUR-PR-01").type(itm_cd="BAT-500").press("Fetch").type(qty="2000", need_dt="09-11-2026", cc_cd="PRD-07").press("Save").press("Submit")
    pr2 = s.form["pr_no"]
    r2 = Desk(http2, t2, "Daniel").sign_in("DOKAFOR", "daniel@123")
    s = r2.open("PUR-PR-04").press("Inbox")
    s.pick(next(i for i, r in enumerate(s.rows) if r["doc_no"] == pr2)).press("Fetch").type(decision="A").press("Confirm")
    c0 = t2.c["codes typed"]
    s = k2.open("PUR-POH-02").type(itm_cd="BAT-500").press("Query").pick(0)
    s = k2.open("PUR-PO-01").type(pr_no=pr2, sup_cd=s.form["sup_cd"], itm_cd="BAT-500").press("Fetch").type(tax_cd="VAT20-RC").press("Save")
    print("careful Save:", s.status, "| status", s.form.get("status"))
    s = k2.open("PUR-POR-05").type(po_no=s.form["po_no"]).press("Fetch").press("Release")
    print("careful Release:", s.status)
    careful_codes = t2.c["codes typed"] - c0
print("codes typed in mission 4, careful path:", careful_codes, "| first time:", before.loc["4 order", "codes typed"])

careful Save: PO/2026/01182 SAVED | status AP
careful Release: PO/2026/01182 RELEASED
codes typed in mission 4, careful path: 6 | first time: 9


## 4. The After: Ask Northbridge

A new, fresh Northbridge, and the same missions, one sentence each. Each person gets their own **Ask Northbridge**,
signed in as themselves, with their own limits. When Ask Northbridge asks a question ("Shall I go ahead?"), the
person answers by fixed rules (`ledgerline.course.l1.answer`), so the run replays exactly. "Done" is read from
Ledgerline's own tables, never from the reply.

For now, treat Ask Northbridge as a black box, like Ledgerline: a sentence goes in, a reply comes out.

In [12]:
from ledgerline.course.l1 import MISSIONS, play_mission

erp3 = ERP()
agents = {}
log = []
for m in MISSIONS:   # the same as play_missions(erp3, agents=agents), keeping each mission's model time
    a = agents.get(m["user"])
    n0 = len(a.llm.log) if a else 0
    r = play_mission(erp3, agents, m)
    r["model_seconds"] = sum(x.seconds for x in agents[m["user"]].llm.log[n0:])   # recorded seconds, kept on replay
    log.append(r)

for m in log:
    display(Markdown(f"### {m['title']} ({m['who']}) · done: **{m['done']}**"))
    for t in m["turns"]:
        display(Markdown(f"> **{m['who']}:** {t['user']}\n\n**Ask Northbridge:** " + (t["reply"] or "").replace("\n", "  \n")))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

### Check the stock (Emma) · done: **True**

> **Emma:** How many city e-bike battery packs do we have, and how many days will they last?

**Ask Northbridge:** We have **6,600** city e-bike battery packs (**BAT-500**, 36 V 500 Wh) in stock:  
  
* **Manchester main stores:** 5,400  
* **Leeds stores:** 1,200  
  
None are currently allocated, and this stock will last for **11 days** of cover.

### Raise the purchase request (Emma) · done: **True**

...

### Receive the rest (Aiko) · done: **True**

> **Aiko:** The other 800 have come in too, delivery note DN-4486.

**Ask Northbridge:** **Goods Receipt Preview (PO/2026/01182 - Voltway Cells)**  
- **Item & Quantity:** 800 × BAT-500 into WH-MAN-STR (completes order, balance 0)  
- **Delivery Note:** DN-4486  
  
Shall I confirm and book this receipt?

> **Aiko:** Yes, go ahead.

**Ask Northbridge:** The goods receipt has been booked under **GRN/2026/30002**.  
  
The final 800 battery packs (BAT-500) have been received into WH-MAN-STR, bringing stock to 7,400. The balance is now 0, and PO/2026/01182 is closed.

### A cheaper supplier? (Emma) · done: **True**

> **Emma:** Brightcell Energy in Lyon quotes £104.00 for the same battery pack. Can we buy from them instead?

**Ask Northbridge:** Not right away. Brightcell Energy is not currently registered in Ledgerline, and battery packs cannot be ordered from them until:  
  
1. **Supplier Onboarding:** They must be formally onboarded with full documentation (POL-PUR-001 §6.3, POL-SUP-003 §3.2).  
2. **UN 38.3 Certification:** As battery packs are lithium-cell parts, we can only purchase from suppliers with a valid UN 38.3 safety certificate checked and on file before an order is placed (POL-PUR-001 §6.2, POL-SUP-003 §4.1).  
  
Additionally, PO/2026/01182 has already been released to Voltway Cells.

Ledgerline, read back after the missions. The documents are the same as in the Before, and nothing is silent: the
order is closed, and the right tax code and cost centre are on it.

In [13]:
display(pd.DataFrame(erp3.q("SELECT pr_no, status, cc_cd, est_value, approved_by FROM pr_hdr WHERE raised_on >= '2026-10-26'")))
display(pd.DataFrame(erp3.q("SELECT po_no, status, cc_cd, tax_cd, gross_value, approved_by FROM po_hdr WHERE created_on >= '2026-10-26'")))
display(pd.DataFrame(erp3.q("SELECT h.grn_no, h.po_no, l.qty, h.received_by, h.remarks FROM grn_hdr h JOIN grn_line l ON l.grn_no = h.grn_no")))
print("Brightcell Energy registered?", erp3.q("SELECT sup_cd FROM supplier WHERE name LIKE '%Brightcell%'") or "no")

,pr_no,status,cc_cd,est_value,approved_by
0,PR/2026/04417,CL,PRD-07,273600.0,DOKAFOR


,po_no,status,cc_cd,tax_cd,gross_value,approved_by
0,PO/2026/01182,CL,PRD-07,VAT20-RC,285600.0,DOKAFOR


,grn_no,po_no,qty,received_by,remarks
0,GRN/2026/30001,PO/2026/01182,1200,ATANAKA,DN-4471
1,GRN/2026/30002,PO/2026/01182,800,ATANAKA,DN-4486


Brightcell Energy registered? no


### The Before and the After, side by side
In the Experience app, a "Yes, go ahead." is one **tap** on the Yes button, not typing. The model's time is what
the recorded run measured (a replay keeps the recorded seconds).

In [14]:
YES = "Yes, go ahead."
A = [m for m in log if m["id"] != "brightcell"]
msgs = [t["user"] for m in A for t in m["turns"]]
typed = [x for x in msgs if x != YES]
model_seconds = sum(m["model_seconds"] for m in A)
after = {"screens opened": 0, "fields typed": 0, "codes typed": sum(x.count("PO/") for x in typed),
         "characters typed": sum(len(x) for x in typed), "buttons pressed": 0, "sign-ins": 0, "error codes": 0}
score = pd.DataFrame({"Before (first time)": before.loc["total", cols], "After (Ask Northbridge)": pd.Series(after)}).astype(object)
score.loc["messages"] = [None, len(msgs)]
score.loc["  of which taps (a yes)"] = [None, len(msgs) - len(typed)]
score.loc["tool calls (Ask Northbridge's work)"] = [None, sum(len(m["cards"]) for m in A)]
score.loc["minutes"] = [before.loc["total", "minutes (estimate)"], round(model_seconds / 60, 1)]
print("Before minutes: hands-on estimate. After minutes: the model's time, measured in the recorded run.")
score

Before minutes: hands-on estimate. After minutes: the model's time, measured in the recorded run.


,Before (first time),After (Ask Northbridge)
screens opened,14,0
fields typed,29,0
codes typed,14,1
characters typed,247,453
buttons pressed,28,0
sign-ins,3,0
error codes,2,0
messages,NaN,12.0
of which taps (a yes),NaN,6.0
tool calls (Ask Northbridge's work),NaN,20.0


### TODO (exercise 3): where did the effort go?
Work out what share of the After's messages were only a yes to a preview, and put it in `yes_share`. Then find
the one mission where Ask Northbridge needed **two** yeses, and say in a sentence why it asked twice.

In [15]:
SOLUTION = True
yes_share = two_yes = None
if SOLUTION:
    yes_share = (len(msgs) - len(typed)) / len(msgs)
    two_yes = [m["title"] for m in A if sum(t["user"] == YES for t in m["turns"]) == 2]
print(f"{yes_share:.0%} of the messages were a yes to a preview; two yeses in: {two_yes}")
# (One yes to create the order, one to release it: an order is placed only when it is RELEASED.)

50% of the messages were a yes to a preview; two yeses in: ['Make and release the order']


## 5. What is behind it

Now open the black box one level. Ask Northbridge is three things:
- a **toolbox**: one tool per business activity, each a folder with `tool.json` (what the LLM reads) and
  `tool.py` (what runs);
- **MCP** (Model Context Protocol), the plug: Ask Northbridge asks the toolbox server what tools it has, then calls them
  through it;
- an **orchestrator LLM** that reads the request, picks the next tool, reads the result, and decides again.

### The tool schemas

In [16]:
from ledgerline.toolbox import SPECS, Toolbox

pd.DataFrame([{"tool": n, "writes": sp.writes, "inputs": ", ".join(sp.spec["input_schema"].get("properties", {})),
               "required": ", ".join(sp.spec["input_schema"].get("required", [])),
               "description": sp.spec["description"][:70]} for n, sp in SPECS.items()])

,tool,writes,inputs,required,description
0,find_item,False,query,query,Find stock items by description or code (e.g. '500 Wh battery pack').
1,check_stock,False,item_code,item_code,"Stock of an item in every warehouse: on hand, allocated, on order, reo"
2,find_supplier,False,"name, city",name,"Look a supplier up by name, tolerating different spellings. Returns th"
3,check_budget,False,cost_centre,,"Is a cost centre open, and how much budget is left this quarter? Witho"
4,search_policy,False,question,question,Search Northbridge's purchasing policy and the Ledgerline manual. Retu
5,create_purchase_request,True,"item_code, quantity, cost_centre, need_by, confirm","item_code, quantity, cost_centre, need_by",Raise a purchase request (PR): the internal ask to buy something. With
6,list_my_approvals,False,,,What is waiting for the signed-in user's signature: purchase requests
7,approve_purchase_request,True,"request_no, decision, comment","request_no, decision",Approve or reject a purchase request that waits for the signed-in user
8,create_purchase_order,True,"supplier_code, request_no, item_code, quantity, cost_centre, need_by, rate, ...",supplier_code,Create a purchase order (PO) with a supplier. Give request_no for an a
9,approve_purchase_order,True,"po_no, comment",po_no,Approve a purchase order that waits for the signed-in user's signature


In [17]:
po_tool = SPECS["create_purchase_order"]
print(json.dumps({k: po_tool.spec[k] for k in ("name", "description", "input_schema")}, indent=1)[:1800])

{
 "name": "create_purchase_order",
 "description": "Create a purchase order (PO) with a supplier. Give request_no for an approved purchase request (it supplies item, quantity, cost centre and date), or item_code, quantity, cost_centre and need_by for a small direct order up to \u00a350,000. confirm=false previews price, tax and value; confirm=true saves it. Returns a receipt with the order's real status: an APPROVED order still has to be released.",
 "input_schema": {
  "type": "object",
  "properties": {
   "supplier_code": {
    "type": "string",
    "description": "Ledgerline supplier code, from find_supplier"
   },
   "request_no": {
    "type": "string",
    "description": "an APPROVED purchase request"
   },
   "item_code": {
    "type": "string",
    "description": "for a direct order"
   },
   "quantity": {
    "type": "integer",
    "description": "for a direct order"
   },
   "cost_centre": {
    "type": "string",
    "description": "for a direct order"
   },
   "need_by": {

In [18]:
print(po_tool.source)

"""Create purchase order: Create a purchase order (PO) with a supplier.

Input and output are described in tool.json next to this file."""
from ledgerline.agent.tools import ServiceTimeout, ToolError, _ledgerline, _receipt, _tax_code, content_key


def run(s, supplier_code, request_no=None, item_code=None, quantity=None, cost_centre=None, need_by=None,
        rate=None, confirm=False):
    if request_no:  # the request carries what to buy, how many, who pays and by when
        pr = _ledgerline(s, "pr_get", pr_no=request_no)
        item_code, quantity = pr["lines"][0]["itm_cd"], pr["lines"][0]["qty"]
        cost_centre, need_by = pr["cc_cd"], pr["need_dt"]
    elif not all([item_code, quantity, cost_centre, need_by]):
        raise ToolError("A direct order needs item_code, quantity, cost_centre and need_by.",
                        "Ask the user, or raise a purchase request first.")
    key = content_key("PO", s.user_id, supplier_code, item_code, quantity, cost_centre, need_by)
  

### TODO (exercise 4): find the traps in the code
Two lines of `tool.py` remove two traps you hit by hand: the wrong tax code (ERR-PO-0412) and the silent Draft on
a closed cost centre. Put those two lines (as text, stripped) in `tax_line` and `draft_line`.

In [19]:
SOLUTION = True
tax_line = draft_line = None
if SOLUTION:
    lines = [l.strip() for l in po_tool.source.splitlines()]
    tax_line = next(l for l in lines if l.startswith("tax_cd = _tax_code"))
    draft_line = next(l for l in lines if l.startswith('if budget["cc_state"] == "CLOSED"'))
print(tax_line)
print(draft_line)

tax_cd = _tax_code(s, supplier_code)  # from the supplier's country, not Ledgerline's default (the ERR-PO-0412 trap)
if budget["cc_state"] == "CLOSED":


### One tool call, by hand
You do not need an LLM to use a tool. `Toolbox(erp, user)` binds the tools to one person's Ledgerline session. As
Emma, on a fresh Northbridge: check the budget of the closed cost centre `PRD-05`, then **preview** (`confirm` is
false by default) a direct order with no request, for 2,000 battery packs.

In [20]:
tb = Toolbox(ERP(), "ECLARKE")
budget = tb.call("check_budget", {"cost_centre": "PRD-05"})
direct = tb.call("create_purchase_order", {"supplier_code": "VC-EIN-0042", "item_code": "BAT-500",
                                           "quantity": 2000, "cost_centre": "PRD-07", "need_by": "2026-11-09"})
show_json(budget, 500)
show_json(direct, 700)

{
 "ok": true,
 "code": "PRD-05",
 "name": "City e-bike line, regular production",
 "open": false,
 "closed_on": "2026-10-15",
 "budget_left": 190000.0,
 "note": "PRD-05 is CLOSED. Ledgerline will silently save any order on it as a DRAFT that never reaches the supplier. Ask the user for an open cost centre."
}
{
 "ok": false,
 "error": "A direct order is limited to £50,000; this one is £285,600.",
 "next_step": "Raise a purchase request first with create_purchase_request.",
 "ledgerline_code": ""
}


Compare this with the Before. There, Ledgerline said `ERR-PO-0610: DIRECT PO LIMIT 50000 EXCEEDED, RAISE PR`. The tool
says what went wrong in words and gives a `next_step`.

### MCP, the plug
MCP is an open protocol, JSON-RPC underneath. A host connects to a server, asks `tools/list`, and then sends
`tools/call` with a tool name and arguments. Our toolbox is an MCP server. Here we connect to it in memory, as Emma,
exactly as Ask Northbridge does.

In [21]:
from ledgerline.mcp_server import MCPClient
from ledgerline.toolbox.mcp import build_toolbox_server

client = MCPClient(build_toolbox_server(ERP(), "ECLARKE"))
listed = client.list_tools()
print(len(listed), "tools over MCP:", [t.name for t in listed])
res = client.call_tool("check_stock", {"item_code": "BAT-500"})
call = client.wire[-1]
print("\nthe request, as a JSON-RPC message:")
show_json({"jsonrpc": "2.0", "id": 2, "method": call["method"], "params": call["params"]})
print("\nthe reply's text content:")
show_json(json.loads(client.text(res)), 600)

15 tools over MCP: ['find_item', 'check_stock', 'find_supplier', 'check_budget', 'search_policy', 'create_purchase_request', 'list_my_approvals', 'approve_purchase_request', 'create_purchase_order', 'approve_purchase_order', 'release_purchase_order', 'order_status', 'receive_goods', 'register_supplier', 'send_message']

the request, as a JSON-RPC message:
{
 "jsonrpc": "2.0",
 "id": 2,
 "method": "tools/call",
 "params": {
  "name": "check_stock",
  "arguments": {
   "item_code": "BAT-500"
  }
 }
}

the reply's text content:
{
 "ok": true,
 "item_code": "BAT-500",
 "description": "Battery pack, 36 V 500 Wh",
 "unit": "NOS",
 "warehouses": [
  {
   "warehouse": "WH-MAN-STR",
   "name": "Manchester main stores",
   "on_hand": 5400,
   "allocated": 0,
   "on_order": 0,
   "free": 5400
  },
  {
   "warehouse": "WH-LDS-STR",
   "name": "Leeds stores",
   "on_hand": 1200,
   "allocated": 0,
   "on_order": 0,
   "free": 1200
  }
 ],
 "free_total": 6600,
 "on_order_total": 0,
 "reorder_level":

### TODO (exercise 5): which tools can change Ledgerline?
Each tool in the `tools/list` reply carries **annotations**. `readOnlyHint` is true for tools that only read.
From `listed`, put the names of the tools that can change Ledgerline in `writers`. A host can use this list to
ask for a yes before any of them runs. Check that it matches `writes` in each `tool.json`.

In [22]:
SOLUTION = True
writers = None
if SOLUTION:
    def read_only(t):  # the field is readOnlyHint on the wire; newer MCP SDKs call it read_only_hint in Python
        d = t.annotations.model_dump() if t.annotations else {}
        return d.get("read_only_hint", d.get("readOnlyHint"))
    writers = sorted(t.name for t in listed if read_only(t) is False)
print(len(writers), "tools can write:", writers)
assert writers == sorted(n for n, sp in SPECS.items() if sp.writes)
client.close()

8 tools can write: ['approve_purchase_order', 'approve_purchase_request', 'create_purchase_order', 'create_purchase_request', 'receive_goods', 'register_supplier', 'release_purchase_order', 'send_message']


And here is what went over the plug when Emma's Ask Northbridge connected in section 4: `tools/list`, then its first
`tools/call`.

In [23]:
emma_an = agents["ECLARKE"]
w = emma_an.client.wire
print("first message on the wire:", w[0]["method"], "->", len(w[0]["result"]["tools"]), "tools")
show_json({"method": w[1]["method"], "params": w[1]["params"]}, 400)
pd.DataFrame([{"mission": m["title"], "who": m["who"], "tool": c["tool"], "arguments": json.dumps(c["args"])[:90],
               "ok": c["result"].get("ok") if isinstance(c["result"], dict) else None}
              for m in log for c in m["cards"]])

first message on the wire: tools/list -> 15 tools
{
 "method": "tools/call",
 "params": {
  "name": "find_item",
  "arguments": {
   "query": "city e-bike battery pack"
  }
 }
}


,mission,who,tool,arguments,ok
0,Check the stock,Emma,find_item,"{""query"": ""city e-bike battery pack""}",True
1,Check the stock,Emma,search_policy,"{""question"": ""city e-bike battery pack""}",True
2,Check the stock,Emma,check_stock,"{""item_code"": ""BAT-500""}",True
3,Check the stock,Emma,check_stock,"{""item_code"": ""BAT-500-R""}",False
4,Raise the purchase request,Emma,check_budget,{},True
5,Raise the purchase request,Emma,check_budget,"{""cost_centre"": ""PRD-07""}",True
6,Raise the purchase request,Emma,create_purchase_request,"{""need_by"": ""2026-11-09"", ""quantity"": 2000, ""confirm"": false, ""cost_centre"":...",True
7,Raise the purchase request,Emma,create_purchase_request,"{""need_by"": ""2026-11-09"", ""cost_centre"": ""PRD-07"", ""quantity"": 2000, ""confir...",True
8,"Approve it, as Daniel",Daniel,list_my_approvals,{},True
9,"Approve it, as Daniel",Daniel,order_status,"{""request_no"": ""PR/2026/04417""}",True


## 6. Your Before (the start of assignment A1)

Every company has a Before like Emma's. It might be in a hospital, a bank, a law firm, a university, a logistics
company or a shop. In A1 you write one from **your own work or domain**. Here is a small structure for it. Each
step names who does it, in which system, and what they must know or type.

### TODO (exercise 6): write your own Before
Replace the example with a process you know. Keep each step short. Mark `code=True` when the person has to type or
remember a code, number or exact name, and `handoff=True` when the work passes to another person. The cell counts
the steps, systems, codes and hand-offs, and the places where the outcome is **silent** (the person cannot see
whether it worked).

In [24]:
SOLUTION = True
my_before = []
if SOLUTION:   # an example from outside manufacturing: a researcher claiming conference travel at a university
    my_before = [
        {"who": "researcher", "system": "travel portal", "step": "find the trip's pre-approval number", "code": True},
        {"who": "researcher", "system": "expenses system", "step": "create a claim, pick the grant code", "code": True},
        {"who": "researcher", "system": "expenses system", "step": "type each receipt and its expense type", "code": True},
        {"who": "researcher", "system": "email", "step": "send scanned receipts to the department office", "handoff": True},
        {"who": "dept. administrator", "system": "expenses system", "step": "check receipts against the grant rules"},
        {"who": "grant holder", "system": "expenses system", "step": "approve the claim", "handoff": True},
        {"who": "finance", "system": "finance ERP", "step": "pay; the claim shows only 'processed'", "silent": True, "handoff": True},
    ]
B = pd.DataFrame(my_before).fillna(False)
print(f"{len(B)} steps · {B['system'].nunique()} systems · {B['who'].nunique()} people · "
      f"{int(B.get('code', pd.Series(dtype=bool)).sum())} codes · {int(B.get('handoff', pd.Series(dtype=bool)).sum())} hand-offs · "
      f"{int(B.get('silent', pd.Series(dtype=bool)).sum())} silent outcomes")
B

7 steps · 4 systems · 4 people · 3 codes · 3 hand-offs · 1 silent outcomes


,who,system,step,code,handoff,silent
0,researcher,travel portal,find the trip's pre-approval number,True,False,False
1,researcher,expenses system,"create a claim, pick the grant code",True,False,False
2,researcher,expenses system,type each receipt and its expense type,True,False,False
3,researcher,email,send scanned receipts to the department office,False,True,False
4,dept. administrator,expenses system,check receipts against the grant rules,False,False,False
5,grant holder,expenses system,approve the claim,False,True,False
6,finance,finance ERP,pay; the claim shows only 'processed',False,True,True


Now write the **one sentence** you wish you could say instead, and what the assistant must **never** do without
asking. A1 asks you for these, and they become your capstone.

In [25]:
my_sentence = "Claim my Lisbon conference trip on the Horizon grant; the receipts are in this folder."   # yours here
never_without_asking = ["submit a claim to a grant it does not belong to", "pay anything"]          # yours here
print("After, in one sentence:", my_sentence)
print("Never without asking:", "; ".join(never_without_asking))

After, in one sentence: Claim my Lisbon conference trip on the Horizon grant; the receipts are in this folder.
Never without asking: submit a claim to a grant it does not belong to; pay anything


## What you did
- **The ERP:** SAP, Oracle, Dynamics and Odoo share one shape: master data, documents with a status, approval
  limits, and code-checking screens. Ledgerline's services are black boxes: an input goes in, an envelope comes out.
- **The Before, measured:** screens, fields, codes, sign-ins, and the traps (ERR-PO-0412, the silent Draft,
  ERR-PO-0715), with an honest estimate of the minutes.
- **The After:** one sentence per mission, a preview and a yes before every save, and receipts read back from
  Ledgerline.
- **What is behind it:** one tool per activity (`tool.json` + `tool.py`), found and called over MCP, and chosen by an
  orchestrator LLM.

**Next time (Lecture 2, The Brain):** we take the tools away and look inside the LLM itself.